![NVIDIA Logo](images/nvidia.png)

# Control Messages in Pipelines

In this notebook you'll create and use a custom stage to operate on a `ControlMessage`.

---

## Objectives

By the time you complete this notebook you will be able to:

- Create custom stages that operate on `ControlMessage` messages.
- Include `ControlMessage` custom stages in pipelines.
- Create a use a re-usable custom stage for annotating control messages with metadata and task details.

---

## Imports

In [1]:
import typing
import logging

from IPython.display import Image

import cudf

from morpheus.config import Config
from morpheus.pipeline import LinearPipeline

from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.output.in_memory_sink_stage import InMemorySinkStage
from morpheus.stages.preprocess.deserialize_stage import DeserializeStage
from morpheus.stages.postprocess.serialize_stage import SerializeStage

from morpheus.messages import MessageMeta, ControlMessage

from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.pipeline.pass_thru_type_mixin import PassThruTypeMixin
from morpheus.pipeline.single_port_stage import SinglePortStage

from morpheus.cli.register_stage import register_stage

from morpheus.utils.logger import configure_logging, reset_logging

import mrc
from mrc.core import operators as ops

---

## Add Metadata and Tasks Helper Function

The following function expects a control message, and returns the control message optionally updated with metadata and tasks.

In [ ]:
def add_metadata_and_tasks(
    cm: ControlMessage,
    metadata: dict = None,
    tasks: dict = None
) -> ControlMessage:
    """
    Adds metadata and/or tasks to a ControlMessage.

    Args:
        cm (ControlMessage): The ControlMessage to modify.
        metadata (dict, optional): Key-value pairs to add as metadata.
        tasks (dict, optional): Tasks to add, where keys are task names and values are task details.

    Returns:
        ControlMessage: The modified ControlMessage.
    """

    # Add metadata if provided
    if metadata:
        for key, value in metadata.items():
            cm.set_metadata(key, value)

    # Add tasks if provided
    if tasks:
        for task_name, task_details in tasks.items():
            cm.add_task(task_name, task_details)

    return cm

### Create a ControlMessage

Here we create a control message, as we've done in previous notebooks.

In [ ]:
input_file = 'data/simple_user_log.jsonlines'

In [ ]:
df = cudf.read_json(input_file, lines=True)

In [ ]:
mm = MessageMeta(df)

In [ ]:
cm = ControlMessage()

In [ ]:
cm.payload(mm)

### Define Metadata and Task Details

The following define metadata and task details which we would like to include as a part of our control message.

In [ ]:
metadata = {
    "priority": "high",
    "source": "api"
}

In [ ]:
task_details = {
    "filter": {
        "IP": "192.168.1.100",
        "port": "443"
    }
}

### Add Metadata and Task Details to Control Message

Now we use our helper function, and the dictionaries defined above, to add metadata and tasks to our control message.

In [ ]:
updated_cm = add_metadata_and_tasks(cm, metadata=metadata, tasks=task_details)

In [ ]:
updated_cm.list_metadata()

In [ ]:
updated_cm.get_metadata('priority')

In [ ]:
updated_cm.get_metadata('source')

In [ ]:
cm.get_tasks()

---

## Create Custom Stage to Annotate Control Message

The following `AddMetadataAndTaskStage` can be used to annotate control messages with metadata and task details.

The custom stage looks very similar to previous custom stages you've created. There is one key change we've made since this stage expects `ControlMessage` type messages, namely, the `accepted_types` method now defines that `ControlMessage` is the expected type.

In [ ]:
@register_stage("add-metadata-and-task")
class AddMetadataAndTaskStage(PassThruTypeMixin, GpuAndCpuMixin, SinglePortStage):
    
    def __init__(self,
                 config: Config,
                 metadata: dict = None,
                 tasks: dict = None
                ):
        super().__init__(config)
        
        self._metadata = metadata
        self._tasks = tasks

    @property
    def name(self) -> str:
        return "add-metadata-and-task"

    # Note that this custom stage expects message type `ControlMessage`.
    def accepted_types(self) -> tuple:
        return (ControlMessage, )

    def supports_cpp_node(self) -> bool:
        return False

    def on_data(self, message: ControlMessage) -> ControlMessage:
        updated_message = add_metadata_and_tasks(message, self._metadata, self._tasks)

        return updated_message

    def _build_single(self, builder: mrc.Builder, input_node: mrc.SegmentObject) -> mrc.SegmentObject:
        node = builder.make_node(self.unique_name, ops.map(self.on_data))
        builder.make_edge(input_node, node)

        return node

---

## Build Pipeline Using Custom Stage

The below pipeline utilizes the `AddMetadataAndTaskStage` just defined. Note that we use our custom stage after `DeserializeStage` so that `ControlMessage` messages are passed into it.

We instantiate `AddMetadataAndTaskStage` with the `metadata` and `task_details` dictionaies we defined earlier in the notebook.

In [ ]:
config = Config()

pipeline = LinearPipeline(config)

pipeline.set_source(FileSourceStage(config, filename=input_file, iterative=False))
pipeline.add_stage(DeserializeStage(config))

pipeline.add_stage(AddMetadataAndTaskStage(config, metadata, task_details))

in_mem_sink = pipeline.add_stage(InMemorySinkStage(config))

pipeline.build()

----

## Run the Pipeline

In [ ]:
viz_file = './pipeline_visualizations/add_metadata.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
await pipeline.run_async()

### Get Control Message

In [ ]:
cm = in_mem_sink.get_messages()[0]

### Get Metadata

In [ ]:
metadata_keys = cm.list_metadata()

In [ ]:
metadata_keys

In [ ]:
for k in metadata_keys:
    print(f"{k}: {metadata.get(k)}")

### Get Tasks

In [ ]:
cm.get_tasks()

In [ ]:
#        (write)                    (read)
# meta   cm.set_metadata          cm.get_metadata
# task   cm.add_task              cm.get_tasks